# Worksheet 3: Let each key $k$ select a diagonal phase operator $D_k$

**Task 3 of 5.** This is the **secret** part of the lock.

```
 |m⟩ ─[F]─► F|m⟩ ─[ D_k ]─► …
                     ▲
            key k ───┘   (picks WHICH D_k)
```

What you'll build, in this order:

1. the number of keys $K$ and what a key is
2. where the key lives (classical or quantum)
3. what $D_k$ must satisfy (and the diagonal gates you already know)
4. how to choose the phase angles (several options, two of them traps)
5. the random codebook with a fixed seed
6. $D_k$ as a matrix
7. $D_k$ as a gate (options + costs)
8. a check of what goes wrong with "structured" phases

### How to use this worksheet

- **📖 Read** cells explain a concept. **✍️ On paper** cells ask you to work something out by hand first.
- **✏️ TODO** cells are yours: replace `None` or the `raise NotImplementedError` with your own code.
- **✅ Check** cells test what you wrote. They print ✅ (correct), ❌ (what is wrong, not how to fix it) or ⏳ (not done yet). **Don't move on until you get ✅.**
- **💡 Hints** are folded. Open them only if you are stuck; Hint 1 is gentle, the last one names the exact function or gate.
- **🔀 Options** tables list the different ways to do a step. Choose one, and write your choice and reason in the decision log at the end, which you can reuse in the report.


## Setup


In [1]:
# Setup (given — nothing to solve here)
# If a package is missing:  !pip install qiskit matplotlib
import numpy as np
import matplotlib.pyplot as plt
import qiskit
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector, Operator, DensityMatrix
from qiskit.circuit.library import QFTGate, DiagonalGate, UnitaryGate

from lock_checks import *     # the automatic checks + plot_circles (keep lock_checks.py in this folder)

np.set_printoptions(precision=3, suppress=True, linewidth=130)
print("Qiskit", qiskit.__version__, "ready")

Qiskit 2.5.2 ready


## Paste your work from worksheets 1–2


In [26]:
# ✏️ paste
N_DEMO, N = 4, 6
d_demo, d, M = 2**N_DEMO, 2**N, 2**N

def bits_of(m, n):
    return [(m >> q) & 1 for q in range(n)]

def ket(m, d):
    v = np.zeros(d, dtype=complex)
    v[m] = 1.0
    return v

def encode_message(m, n):
    qr = QuantumRegister(n, "msg")
    qc = QuantumCircuit(qr)
    for q in range(n):
        if (m >> q) & 1:
            qc.x(qr[q])
    return qc

def dft_matrix(n):
    d = 2 ** n
    x = np.arange(d)
    return np.exp(2j * np.pi * np.outer(x, x) / d) / np.sqrt(d)

In [27]:
check_dims(N_DEMO, N, d_demo, d, M)
check_ket(ket)
check_encode(encode_message)
check_dft(dft_matrix)

✅ dimensions d and M — d_demo = 16, d = 64, M = 64
✅ ket(m, d) — length, normalization and position all correct
✅ encode_message(m, n) — every message prepares the right basis state (n = 3 and 6)
✅ dft_matrix(n) — matches the definition for n = 1..6


True

---

## Step 3.1: What is a key, and how many?

📖 A key is an integer $k\in\{0,\dots,K-1\}$ shared secretly by Alice and Bob, so it is $\log_2K$ bits long.

The whole point of **data locking** (paper Sec. I–II) is that the key can be much **shorter** than the message: $K\ll M$, ideally $\log K = O(\log\log M)$. A classical one-time pad needs a key as long as the message.

🔀 **Options**

| $K$   | key bits | message bits ($n=6$) | comment                                     |
| ----- | -------- | -------------------- | ------------------------------------------- |
| 2     | 1        | 6                    | smallest interesting case                   |
| 4     | 2        | 6                    | brief's suggestion                          |
| 8, 16 | 3, 4     | 6                    | for the "vary $K$" experiments later        |
| 64    | 6        | 6                    | as long as the message: no longer "locking" |

✍️ For $n=6$, compute $\log_2\log_2 M$. Which $K$ is "in the spirit" of the paper?


In [28]:
# ✏️ TODO 3.1
K = 4
keys = list(range(K))

In [29]:
check_K(K, keys)

✅ number of keys — K = 4 -> 2 key bits


True

## Step 3.2: Where does the key live? (decide, no code)

🔀 **Options**

| Option | Implementation                                                       | Pros                                                                                                   | Cons                                             |
| ------ | -------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------ | ------------------------------------------------ |
| A      | a **classical** Python integer that decides _which_ gate is appended | brief: "the key may remain a classical input selecting one of several phase circuits"; only $n$ qubits | one circuit per $(m,k)$, which is fine           |
| B      | a **quantum** key register + controlled-$D_k$                        | can model "unknown key" as a superposition                                                             | more qubits, many controlled gates, not required |
| C      | classical bits + dynamic circuits (`if_test`)                        | hardware-style                                                                                         | `Statevector` can't simulate it                  |

✍️ Write your choice in the decision log. (The checks in worksheets 4–5 assume A.)


Option A, classical key. The key is a Python integer
k
∈
{
0
,
…
,
K
−
1
}
k∈{0,…,K−1} that selects which phase circuit to append. The brief explicitly allows this, and it keeps the register purely quantum with only
n
n qubits — matching what Worksheets 4–5 assume.


## Step 3.3: What must $D_k$ satisfy?

$$D_k=\mathrm{diag}\big(e^{i\theta_{k,0}},\dots,e^{i\theta_{k,d-1}}\big),\qquad D_k|x\rangle=e^{i\theta_{k,x}}|x\rangle .$$

✍️ **Think about each requirement and write in your notes _why_ it's needed:**

| Requirement                                                          | Why? (your words) |
| -------------------------------------------------------------------- | ----------------- |
| diagonal                                                             |                   |
| every entry has modulus 1                                            |                   |
| different for each key                                               |                   |
| phases "random-looking", $\mathbb E[e^{i\theta}]=0$ (paper Eq. 1/16) |                   |

📖 **You already know diagonal gates from the course.** Get the diagonal of each matrix below and look at them. $D_k$ is the most general version: one free phase per basis state.


In [31]:
# ✏️ TODO 3.3: diagonals of the course gates (Operator(...).data, then np.diag)
from qiskit.circuit.library import ZGate, SGate, TGate, CZGate, CPhaseGate

diag_Z  = np.diag(Operator(ZGate()).data)
diag_S  = np.diag(Operator(SGate()).data)
diag_T  = np.diag(Operator(TGate()).data)
diag_CZ = np.diag(Operator(CZGate()).data)

In [32]:
check_course_diagonals(diag_Z, diag_S, diag_T, diag_CZ)

✅ diagonals of the course gates


True

Each is a vector of modulus‑1 entries:
Z
=
d
i
a
g
(
1
,
−
1
)
Z=diag(1,−1),
S
=
d
i
a
g
(
1
,
i
)
S=diag(1,i),
T
=
d
i
a
g
(
1
,
e
i
π
/
4
)
T=diag(1,e
iπ/4
),
C
Z
=
d
i
a
g
(
1
,
1
,
1
,
−
1
)
CZ=diag(1,1,1,−1).
D
k
D
k
​
is the general version: one independent phase per basis state.


<details><summary>💡 Hint 1</summary>

`np.diag(matrix)` returns the diagonal of a matrix (and `np.diag(vector)` builds a diagonal matrix: you'll need both).

</details>


## Step 3.4: How to choose the angles $\theta_{k,x}$ (decide)

🔀 **Options**

| Option | Angles                            | $D_k$ entries | Source                                                                            | Pros                                             | Cons                                                           |
| ------ | --------------------------------- | ------------- | --------------------------------------------------------------------------------- | ------------------------------------------------ | -------------------------------------------------------------- |
| A      | random $\{0,\pi\}$                | $\pm1$        | paper: "even $(-1)^{b(\omega)}$ is sufficient"; brief: "simpler first experiment" | simple, links to the course _flip / phase-logic_ | $D_k=D_k^\dagger$, which can hide a bug in worksheet 5         |
| B      | random uniform in $[0,2\pi)$      | $e^{i\theta}$ | brief: main recipe                                                                | the full phase ensemble                          | the gate is expensive (Step 3.8)                               |
| C      | random from $\{0,90°,180°,270°\}$ | $1,i,-1,-i$   | between A and B                                                                   |                                                  |                                                                |
| D      | linear $\theta_x=2\pi sx/d$       |               |                                                                                   | very cheap                                       | ⚠️ trap: you'll test it in Step 3.9                            |
| E      | constant $\theta_x=c$             |               |                                                                                   |                                                  | ⚠️ trap: what does $e^{ic}I$ do to a state?                    |
| F      | quadratic $\theta_x=2\pi a x^2/d$ |               |                                                                                   | cheap circuit                                    | structured, not the paper's ensemble (an extension to discuss) |

Recommendation: support **A and B** with a `mode` switch.


## Step 3.5: The codebook, random but reproducible

📖 The codebook is all $K\times d$ angles. Alice and Bob must generate **the same** one, and you must be able to rerun experiments, so use a **fixed seed**.

🔀 **Options for randomness**

| Option | Code idea                                                                        | Comment                                                |
| ------ | -------------------------------------------------------------------------------- | ------------------------------------------------------ |
| A      | `rng = np.random.default_rng(seed)` then `rng.uniform(...)`, `rng.integers(...)` | modern, independent of other random calls              |
| B      | `np.random.seed(seed)` + global functions                                        | any other random call in between changes your codebook |
| C      | also save the angles to a file (`np.savez`)                                      | exact reproducibility for the report                   |

Write `make_codebook(n, K, seed, mode)` returning an array of shape `(K, d)`:

- `mode="uniform"`: angles uniform in $[0,2\pi)$
- `mode="binary"`: angles in $\{0,\pi\}$


In [33]:
# ✏️ TODO 3.5
def make_codebook(n, K, seed=2026, mode="uniform"):
    d = 2 ** n
    rng = np.random.default_rng(seed)
    if mode == "uniform":
        return rng.uniform(0.0, 2.0 * np.pi, size=(K, d))
    elif mode == "binary":
        return np.pi * rng.integers(0, 2, size=(K, d))
    else:
        raise ValueError(f"unknown mode {mode!r}")

In [34]:
check_codebook(make_codebook)

✅ make_codebook(n, K, seed, mode) — shape, range, reproducibility and both modes OK


True

<details><summary>💡 Hint 1</summary>

Create the generator inside the function from `seed`, so the same seed always gives the same angles.

</details>

<details><summary>💡 Hint 2</summary>

`rng.uniform(low, high, size=(K, d))`; for binary: `np.pi * rng.integers(0, 2, size=(K, d))`.

</details>


Now print the codebook for `N_DEMO` (in degrees, rounded) for both modes. Then check the paper's condition $\mathbb E[e^{i\theta}]\approx 0$: compute $|\text{mean of } e^{i\theta}|$ over the whole project-size codebook. How close to 0 is it? Compare with $1/\sqrt{Kd}$.


In [35]:
# ✏️ TODO 3.5b: print the demo codebooks; compute |mean(e^{i theta})| for N
for mode in ("uniform", "binary"):
    cb = make_codebook(N_DEMO, K, 2026, mode)
    print(f"--- demo codebook ({mode}), degrees ---")
    print(np.round(np.degrees(cb), 1))

thetas_full = make_codebook(N, K, 2026, "uniform")
mean_phase = np.abs(np.mean(np.exp(1j * thetas_full)))
print("\n|mean(e^{i theta})| =", mean_phase)
print("1/sqrt(K*d) =", 1 / np.sqrt(K * d))

--- demo codebook (uniform), degrees ---
[[ 64.4 230.4 168.2 133.4 127.8 284.6 325.9  63.8 235.  107.4 348.1 331.1 228.9 271.  185.5 297.3]
 [161.4 122.  100.   81.5 189.3 155.1 238.7   4.6 161.2 131.5  70.3 214.2 156.7 108.   75.4 314.9]
 [287.1 218.4 124.2 340.9 202.8 155.8 324.2 115.  250.6 113.   94.2 252.3  82.  177.5 208.8  68. ]
 [263.2 197.5 223.7 134.  151.3 178.1 169.2 243.2 207.8 149.9   0.6 285.9 187.  117.6 180.   33.6]]
--- demo codebook (binary), degrees ---
[[180.   0.   0. 180.   0.   0.   0.   0. 180.   0. 180. 180. 180. 180. 180.   0.]
 [180. 180.   0.   0.   0. 180. 180. 180.   0. 180. 180. 180.   0. 180. 180. 180.]
 [180.   0.   0.   0.   0.   0.   0.   0. 180. 180.   0.   0.   0. 180. 180.   0.]
 [  0.   0. 180.   0.   0.   0.   0. 180.   0.   0. 180.   0. 180.   0. 180. 180.]]

|mean(e^{i theta})| = 0.08285403664859288
1/sqrt(K*d) = 0.0625


The empirical mean is close to 0 (within the same order as
1
/
K
d
1/
Kd
​
), confirming the paper's condition
E
[
e
i
θ
]
=
0
E[e
iθ
]=0.


## Step 3.6: $D_k$ as a matrix

🔀 **Options**

| Option | Idea                                                                                            |
| ------ | ----------------------------------------------------------------------------------------------- |
| A      | a full $d\times d$ diagonal matrix                                                              |
| B      | keep only the vector of diagonal entries and multiply element-wise (faster for big loops later) |

Write A first.


In [36]:
# ✏️ TODO 3.6a: D_k = diag(e^{i theta_k,0}, ..., e^{i theta_k,d-1})
# ✏️ TODO 3.6a
def D_matrix(theta_row):
    return np.diag(np.exp(1j * np.asarray(theta_row)))

In [37]:
check_D_matrix(D_matrix)

✅ D_matrix(theta_row) — diagonal with entries e^{i theta}


True

**3.6b.** Pick `thetas_demo = make_codebook(N_DEMO, K, 2026, "uniform")` and key `k = 2`. Compute $D_k^\dagger$ **in two ways**, as the conjugate transpose and as `D_matrix` of the negated angles, and convince yourself they are equal. Also check $D_k^\dagger D_k=I$. Then repeat with a **binary** codebook: what do you notice about $D_k$ vs $D_k^\dagger$?


In [38]:
# ✏️ TODO 3.6b
# ✏️ TODO 3.6b
thetas_demo = make_codebook(N_DEMO, K, 2026, "uniform")
k = 2
Ddag = D_matrix(thetas_demo[k]).conj().T

In [39]:
check_D_dagger(Ddag, thetas_demo[k] if thetas_demo is not None else None)

✅ D_k^dagger


True

## Step 3.7: $D_k$ as a gate

🔀 **Options**

| Option | Code idea                                                           | Pros                                                                 | Cons                                  |
| ------ | ------------------------------------------------------------------- | -------------------------------------------------------------------- | ------------------------------------- |
| A      | `DiagonalGate(list_of_complex_entries)`                             | recommended in the brief; efficient (~$2^n$ CNOTs); has `.inverse()` | black box                             |
| B      | `UnitaryGate(your_D_matrix)`                                        | same matrix                                                          | generic synthesis, far more expensive |
| C      | manual: X gates + multi-controlled PHASE per basis state (Step 3.8) | only course gates; very instructive                                  | very expensive; only small $n$        |

⚠️ Check that entry $j$ of the list acts on basis state $|j\rangle$ in Qiskit's little-endian numbering. The check below does that for you.


In [40]:
# ✏️ TODO 3.7: return a Qiskit gate whose matrix is D_matrix(theta_row)
# ✏️ TODO 3.7
def D_gate(theta_row):
    entries = np.exp(1j * np.asarray(theta_row))
    return DiagonalGate(list(entries))

In [41]:
check_D_gate(D_gate)

✅ D_gate(theta_row) — gate matrix = diag(e^{i theta}) in Qiskit's ordering


True

<details><summary>💡 Hint 1</summary>

`DiagonalGate` wants the **complex diagonal entries**, not the angles.

</details>

<details><summary>💡 Hint 2</summary>

`DiagonalGate(list(np.exp(1j * np.asarray(theta_row))))`

</details>


## Step 3.8 (optional, deeper): Build $D_k$ from course gates

📖 Idea (the course's _flip_ primitive from amplitude amplification): to add phase $\theta_x$ **only** to basis state $|x\rangle$:

1. apply X to every qubit whose bit in $x$ is **0**, so $|x\rangle$ becomes $|11\dots1\rangle$
2. apply a multi-controlled PHASE($\theta_x$), which only affects $|11\dots1\rangle$ (like CPHASE / CZ, deck 17.B)
3. undo step 1

Do this for every $x$. For the binary mode, PHASE(π) is a multi-controlled Z.

✍️ Before coding: for $n=3$, how many multi-controlled gates do you need in the worst case?


In [42]:
# ✏️ TODO 3.8 (optional)
def diagonal_manual(theta_row, n):
    qc = QuantumCircuit(n)
    for x, theta in enumerate(theta_row):
        # skip the identity entries
        if np.isclose(theta % (2 * np.pi), 0.0, atol=1e-12):
            continue
        # 1. flip qubits where bit q of x is 0, so |x> -> |11...1>
        flips = [q for q in range(n) if ((x >> q) & 1) == 0]
        for q in flips:
            qc.x(q)
        # 2. multi-controlled phase
        qc.mcp(theta, list(range(n - 1)), n - 1)
        # 3. undo
        for q in flips:
            qc.x(q)
    return qc

In [43]:
check_diagonal_manual(diagonal_manual)

✅ diagonal_manual(theta_row, n) — built from X + (multi-)controlled phases


True

For
n
=
3
n=3, worst case is
2
n
−
1
=
7
2
n
−1=7 multi-controlled phases (state
x
=
0
x=0 also gets a phase, but skip it if
θ
=
0
θ=0). One circuit per basis state, so it is
O
(
2
n
)
O(2
n
) multi-controlled gates.


<details><summary>💡 Hint 1</summary>

Multi-controlled phase in Qiskit: `qc.mcp(angle, control_qubits_list, target_qubit)`.

</details>

<details><summary>💡 Hint 2</summary>

Controls `list(range(n-1))`, target `n-1`. Skip $x$ where the angle is 0.

</details>


**Costs.** For $n=2..6$, transpile a circuit with your `D_gate` (and, if you did it, `diagonal_manual`) using `basis_gates=["cx","rz","sx","x"]` and count `cx`. Compare with the QFT numbers from worksheet 2. Which part dominates for large $n$? (This is why the brief says 4–8 qubits.)


In [44]:
# ✏️ TODO 3.8b: CNOT-count table
# ✏️ TODO 3.8b
for n in range(2, 7):
    row = make_codebook(n, 1, 2026, "uniform")[0]
    qc = QuantumCircuit(n)
    qc.append(D_gate(row), range(n))
    tqc = transpile(qc, basis_gates=["cx", "rz", "sx", "x"])
    ops = tqc.count_ops()
    print(f"n={n}: cx={ops.get('cx', 0)}, total={sum(ops.values())}")

n=2: cx=2, total=5
n=3: cx=6, total=13
n=4: cx=14, total=29
n=5: cx=30, total=61
n=6: cx=62, total=125


The general diagonal costs about
2
n
2
n
CNOTs, whereas the QFT is
O
(
n
2
)
O(n
2
). For
n
≥
6
n≥6,
D
k
D
k
​
dominates.


## Step 3.9: The traps, and why the phases must be _random_

✍️ **On paper first.** Take the linear phase $\theta_x=2\pi sx/d$. Write out $D\,F|m\rangle$ using the formula of $F|m\rangle$ from worksheet 2, and combine the two exponentials. What state do you get?

Then test it numerically with $n=3$, $s=3$, $m=1$: is $D_{\text{lin}}F|m\rangle$ equal to $F|m+s \bmod d\rangle$?


In [45]:
# ✏️ TODO 3.9: True if D_linear F|1> == F|1+3 mod 8>
# ✏️ TODO 3.9
n_test, s = 3, 3
d_test = 2 ** n_test
m_test = 1
F_test = dft_matrix(n_test)
theta_lin = 2 * np.pi * s * np.arange(d_test) / d_test
lhs = D_matrix(theta_lin) @ F_test @ ket(m_test, d_test)
rhs = F_test @ ket((m_test + s) % d_test, d_test)
is_shift = bool(np.allclose(lhs, rhs))

In [46]:
check_linear_trap(is_shift)

✅ linear-phase trap — a linear 'key' just shifts the message


True

🔬 Extra: play Eve with **no key**: apply $F^\dagger$ to $D_{\text{lin}}F|1\rangle$ and print the probabilities. Do the same with a random $D_k$ from your codebook. Explain the difference in one sentence (Caesar cipher?).

✍️ And trap E: what does a constant phase $\theta_x=c$ do?


In [47]:
# ✏️ TODO 3.9b
F3 = dft_matrix(n_test)

# (i) linear phase: F^† D_lin F |m> is just |m+s>, Eve reads it directly
eve_lin_state = F3.conj().T @ D_matrix(theta_lin) @ F3 @ ket(m_test, d_test)
print("linear trap, F^† D_lin F |1> =", np.round(eve_lin_state.real, 3))

# (ii) random phase: F^† D_k F |m> is NOT a basis state
thetas_rand = make_codebook(n_test, K, 2026, "uniform")
k_rand = 1
rand_state = F3.conj().T @ D_matrix(thetas_rand[k_rand]) @ F3 @ ket(m_test, d_test)
probs_rand = np.abs(rand_state) ** 2
print("random key, probabilities:", np.round(probs_rand, 3))

linear trap, F^† D_lin F |1> = [-0. -0.  0. -0.  1. -0.  0. -0.]
random key, probabilities: [0.21  0.17  0.036 0.002 0.097 0.086 0.168 0.231]


Explanation. The linear case is a classical shift (Caesar cipher): once Eve learns
s
s by any side channel, the message is trivially recovered. The random case leaves the phase spread over all basis states with no structure — Eve cannot recover
m
m without knowing
k
k.

Constant-phase trap. A constant
θ
x
=
c
θ
x
​
=c gives
D
=
e
i
c
I
D=e
ic
I, a pure global phase. It does nothing observable.


## Step 3.10: How different are the keys?

📖 For two keys, define the overlap $\left|\frac1d\sum_x e^{i(\theta_{k,x}-\theta_{k',x})}\right|$. It is 1 for the same key and small for random keys (about $1/\sqrt d$). A small overlap means a wrong key leaves lots of random phase behind, which matters for worksheet 6.

Compute the $K\times K$ matrix of overlaps for the project codebook (`N`, uniform).


In [48]:
# ✏️ TODO 3.10
thetas = make_codebook(N, K, 2026, "uniform")
E = np.exp(1j * thetas)              # K x d
ov = np.abs(E @ E.conj().T) / d      # K x K

In [49]:
check_overlap(ov, thetas)

✅ overlap matrix between keys


True

Diagonal entries equal 1 (same key), off‑diagonal entries are close to
1
/
d
1/
d
​
.


<details><summary>💡 Hint 1</summary>

Stack $e^{i\theta}$ as a $K\times d$ matrix $E$. Then $E E^\dagger / d$ holds all the sums at once; take `np.abs`.

</details>


## Step 3.11: Freeze the project codebook

Choose `SEED` and `MODE`, generate `thetas` for `N` and `K`, and (option C of Step 3.5) save it with `np.savez("codebook.npz", ...)`. Worksheets 4–5 regenerate it from the same seed.


In [50]:
# ✏️ TODO 3.11
SEED = 2026
MODE = "uniform"
thetas = make_codebook(N, K, SEED, MODE)
np.savez("codebook.npz", thetas=thetas, K=K, N=N, seed=SEED, mode=MODE)
print("saved codebook.npz:", thetas.shape)

saved codebook.npz: (4, 64)


---

## Self-check questions

1. Why can't $D_k$ change measurement probabilities in the computational basis?
2. Why is a linear phase useless as a key?
3. In binary mode, what is $D_k^\dagger$?
4. Why does the key's gate cost more than the QFT for large $n$?

<details><summary>Answers</summary>

1. It multiplies each amplitude by a number of modulus 1.
2. It just shifts the message ($F|m\rangle\to F|m+s\rangle$).
3. $D_k$ itself.
4. A general diagonal has $d=2^n$ independent phases (~$2^n$ CNOTs), while the QFT is $O(n^2)$.
</details>

## Decision log

| Step           | Options                             | My choice | Why |
| -------------- | ----------------------------------- | --------- | --- |
| 3.1 $K$        | 2 / 4 / 8 / 16                      |           |     |
| 3.2 key        | classical / quantum / dynamic       |           |     |
| 3.4 angles     | binary / uniform / q-ary / chirp    |           |     |
| 3.5 randomness | default_rng / global seed / + file  |           |     |
| 3.7 gate       | DiagonalGate / UnitaryGate / manual |           |     |

✅ All green? Continue with **Worksheet 4 (locked state)**.
